# Lab 4: LLMs and Prompt Engineering for Decision Support




**Student Name:** Mohamet Aloula Na'ima

**Student ID:** 30722027


### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [ ]:
from google.colab import userdata
from google import genai
from google.genai import types

# Get the API key securely from Colab Secrets
API_KEY = userdata.get("GeminiAPIKey4")

# Create Gemini client
client = genai.Client(api_key=API_KEY)

print("Gemini client ready.")

Gemini client ready.


In [ ]:
for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.6-preview
models/gemini-robotics-er-2-preview
models/gemini-2.5-computer-use-preview-10-2025
models/an

In [ ]:
MODEL = "models/gemini-3.1-flash-lite"  # explicit lite model, higher free-tier daily quota


In [ ]:
# Sanity check only — never print the key itself
print("API key loaded:", API_KEY is not None)


API key loaded: True


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [ ]:
def ask_llm(
    user_prompt,
    system_prompt="You are a helpful assistant.",
    temperature=0.7,
    max_tokens=500
):
    response = client.models.generate_content(
        model=MODEL,
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=temperature,
            max_output_tokens=max_tokens
        )
    )
    return response

**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:**
1) The system role gives the model general instructions about how it should behave, while the user role contains the specific request or question. For example, the system message can say “You are a helpful assistant,” while the user message can ask “What is a microfinance loan?”

2) A token is a small unit of text that an LLM processes. It can be a whole word, part of a word, or punctuation. API providers charge based on tokens because the amount of text the model processes and generates determines the computational resources used, whereas different requests can contain very different amounts of text.



### Part 1.2 — Temperature: the randomness dial

In [ ]:
question = "Suggest a name for a savings product for market traders in Accra."

print("=== Temperature 0.0 (5 runs) ===")
for i in range(5):
    response = ask_llm(question, temperature=0.0, max_tokens=60)
    print(f"{i+1}: {response.text.strip()}")

print("\n=== Temperature 1.2 (5 runs) ===")
for i in range(5):
    response = ask_llm(question, temperature=1.2, max_tokens=60)
    print(f"{i+1}: {response.text.strip()}")


=== Temperature 0.0 (5 runs) ===
1: To choose the right name for a savings product in Accra, you need to balance **trust, growth, and cultural relevance**. Market traders (often called *market queens* or *kayayei* depending on the role) value security, accessibility, and the ability to grow their
2: To choose the right name for a savings product in Accra, you need to balance **trust, growth, and cultural relevance**. Market traders (often called *market queens* or *kayayei* depending on the role) value security, accessibility, and the ability to grow their
3: To choose the right name for a savings product in Accra, you need to balance **trust, growth, and cultural relevance**. Market traders (often called *market queens* or *kayayei* depending on the role) value security, accessibility, and the ability to grow their
4: To choose the right name for a savings product in Accra, you need to balance **trust, growth, and cultural relevance**. Market traders (often called *market queens* or *

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:**

At low temperature, the responses were more consistent. At higher temperatures, the answers became more random. For the loan decision-support system, low temperature is appropriate because we need consistent and reliable decisions.


---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [ ]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [ ]:
SUMMARY_PROMPT_V1 = "Summarize this: {letter}"

for letter_id in ["L002", "L006"]:
    prompt = SUMMARY_PROMPT_V1.format(letter=LETTERS[letter_id])
    response = ask_llm(prompt, temperature=0)
    print(f"--- {letter_id} (V1) ---")
    print(response.text.strip(), "\n")



--- L002 (V1) ---
Here is a summary of the request:

Kwame Boateng, a commercial driver in Kumasi, is requesting an urgent loan of GHS 25,000 to cover vehicle repairs and personal debts. He currently lacks collateral and intends to repay the loan once business improves after the festive season. 

--- L006 (V1) ---
Here is a summary of the request:

Kofi, a 22-year-old entrepreneur, is seeking a GHS 50,000 loan to launch three separate ventures: a car wash, a provision shop, and a phone importation business. He has no prior experience or collateral but promises to repay the loan within one year, citing his "business-minded" reputation as his primary guarantee. 



In [ ]:
SUMMARY_SYSTEM_V2 = """You are an assistant to a microfinance loan officer in Ghana.
Summarize loan application letters factually and neutrally in 3-4 sentences.
Do not invent, guess, or add any detail that is not explicitly stated in the letter."""

SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter}"

for letter_id in ["L002", "L006"]:
    prompt = SUMMARY_PROMPT_V2.format(letter=LETTERS[letter_id])
    response = ask_llm(prompt, system_prompt=SUMMARY_SYSTEM_V2, temperature=0)
    print(f"--- {letter_id} (V2) ---")
    print(response.text.strip(), "\n")

--- L002 (V2) ---
Kwame Boateng, a commercial driver based in Kumasi, is requesting a loan of GHS 25,000. He intends to use the funds to repair his vehicle's engine and settle personal debts. The applicant states that he currently lacks collateral and proposes to repay the loan once his business improves after the festive season. 

--- L006 (V2) ---
Kofi, a 22-year-old applicant, is requesting a loan of GHS 50,000 to establish a car washing business, a provision shop, and a phone importation venture. He states that he has not yet started these businesses and currently has no collateral to offer. He proposes to repay the loan within one year, citing his personal trustworthiness as a guarantee. 



**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:**

1. **V1 problems:**
V1 included extra or incorrect details that were not in the original text. V2 fixed this by staying closer to the source and removing invented information.

2. **Why “no invented details” matters:**
In a loan system, made-up information could lead to an incorrect loan decision and unfair results. This failure is called **hallucination** in LLM literature.


### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [ ]:
import json
import pandas as pd

EXTRACT_SYSTEM = """You extract structured data from microfinance loan application letters.
Return ONLY a JSON object with exactly these keys:
applicant_name (string), amount_ghs (number), purpose (string),
monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean), repayment_months (number or null).
If a field is not stated in the letter, use null. Do not guess."""

EXTRACT_EXAMPLE = """Example letter:
"My name is Ama Serwaa. I run a small chop bar in Tema and need GHS 6000 to renovate my kitchen.
I make about GHS 700 profit a month. My brother will guarantee the loan. I can pay GHS 400 monthly for 15 months."

Example output:
{"applicant_name": "Ama Serwaa", "amount_ghs": 6000, "purpose": "renovate kitchen",
"monthly_profit_ghs": 700, "has_collateral_or_guarantor": true, "repayment_months": 15}"""

def extract_fields(letter_text):
    # NOTE: built with string concatenation (+), not .format() -- the JSON braces
    # in EXTRACT_EXAMPLE would otherwise confuse .format() into a KeyError.
    prompt = EXTRACT_EXAMPLE + "\n\nNow extract from this letter:\n\n" + letter_text
    response = ask_llm(prompt, system_prompt=EXTRACT_SYSTEM, temperature=0)
    raw = response.text.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        print(f"Warning: could not parse JSON:\n{raw}")
        return None

rows = []
for letter_id, letter_text in LETTERS.items():
    result = extract_fields(letter_text)
    if result:
        result["letter_id"] = letter_id
        rows.append(result)

extracted_df = pd.DataFrame(rows).set_index("letter_id")
extracted_df


,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
letter_id,,,,,,
L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
L002,Kwame Boateng,25000,repair trotro engine and settle personal debts,NaN,False,NaN
L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
L004,Yaw Owusu,12000,feed and 500 new layers,1500.0,True,18.0
L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn,NaN,True,16.0
L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:**

1. The example should not come from the six letters because it could make the model memorize the answers instead of learning the extraction task.

2. Without “use null, do not guess,” the model sometimes **guessed missing information** instead of leaving the field empty.

3. Temperature = 0 is best for extraction because we want **consistent and accurate results**. For creative tasks, a higher temperature can be better because it gives more varied and creative responses.


### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [ ]:
BRIEF_SYSTEM = """You are an assistant to a microfinance loan officer in Ghana.
Given a loan application letter and its extracted data, produce a decision-support brief with:
1. Strengths (bullet points, grounded only in the letter)
2. Risks / red flags (bullet points)
3. Missing information the officer should request
4. Suggested next step (e.g. "invite for interview", "request documents", "flag for senior review")
Never output "approve" or "reject" — final decisions are made by a human loan officer, not you."""

BRIEF_PROMPT = "Letter:\n{letter}\n\nExtracted data:\n{extracted}\n\nProduce the brief."

def generate_brief(letter_id):
    extracted = extracted_df.loc[letter_id].to_dict() if letter_id in extracted_df.index else {}
    prompt = BRIEF_PROMPT.format(letter=LETTERS[letter_id], extracted=json.dumps(extracted))
    response = ask_llm(prompt, system_prompt=BRIEF_SYSTEM, temperature=0)
    return response.text.strip()

briefs = {letter_id: generate_brief(letter_id) for letter_id in LETTERS}

for letter_id in ["L001", "L002", "L006"]:
    print(f"=== Brief for {letter_id} ===")
    print(briefs[letter_id], "\n")

=== Brief for L001 ===
### Loan Application Decision-Support Brief: Akosua Mensah

**1. Strengths**
*   **Proven Business Longevity:** The applicant has operated a business at Makola Market for 12 years, indicating stability and market experience.
*   **Strong Savings History:** The applicant has a two-year track record with the institution’s *susu* scheme and has maintained a consistent contribution history.
*   **Clear Business Purpose:** The loan is tied to a specific asset (deep freezer) intended to diversify revenue streams.
*   **Guarantor Identified:** The applicant has secured a guarantor (a teacher), which provides an additional layer of security for the loan.

**2. Risks / Red Flags**
*   **Repayment-to-Income Ratio:** The proposed monthly repayment of GHS 450 represents 50% of the applicant's current monthly profit (GHS 900). This leaves very little margin for error, unexpected business expenses, or personal emergencies.
*   **Unverified Revenue:** The profit figure of GHS 9

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:**


1. Yes, the system identified the main strengths and red flags correctly. **L003** showed strong business experience and savings, while **L006** had major risks such as no business experience, no collateral, and unrealistic plans.

2. We forbid “approve/reject” because:

    **Practical:** The system should support the loan officer, not make the final decision.

    **Ethical:** A final automated decision could be unfair or biased and may affect someone’s access to credit.


### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [paste here]

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [ ]:
fields = ["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs",
          "has_collateral_or_guarantor", "repayment_months"]

accuracy_rows = []
for field in fields:
    row = {"field": field}
    correct = 0
    for letter_id in ["L001", "L003", "L006"]:
        gold_val = GOLD[letter_id][field]
        pred_val = extracted_df.loc[letter_id, field] if letter_id in extracted_df.index else None
        if field == "applicant_name" and isinstance(gold_val, str) and isinstance(pred_val, str):
            match = gold_val.strip().lower() == pred_val.strip().lower()
        else:
            match = gold_val == pred_val
        row[letter_id] = "✓" if match else f"✗ (got {pred_val})"
        correct += match
    row["accuracy"] = f"{correct}/3"
    accuracy_rows.append(row)

pd.DataFrame(accuracy_rows).set_index("field")

,L001,L003,L006,accuracy
field,,,,
applicant_name,✓,✓,✓,3/3
amount_ghs,✓,✓,✓,3/3
purpose,✗ (got buy a deep freezer and expand into froz...,✗ (got purchase two industrial sewing machines...,"✗ (got start a car washing business, a provisi...",0/3
monthly_profit_ghs,✓,✓,✗ (got nan),2/3
has_collateral_or_guarantor,✓,✓,✓,3/3
repayment_months,✓,✓,✓,3/3


### Part 4.2 — Reliability: is the system consistent?

In [ ]:
def extract_fields_temp(letter_text, temperature):
    # same fix as extract_fields: concatenation, not .format()
    prompt = EXTRACT_EXAMPLE + "\n\nNow extract from this letter:\n\n" + letter_text
    response = ask_llm(prompt, system_prompt=EXTRACT_SYSTEM, temperature=temperature)
    raw = response.text.strip().replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return None

for temp in [0, 1.0]:
    print(f"--- Temperature {temp} ---")
    outputs = [extract_fields_temp(LETTERS["L004"], temp) for _ in range(5)]
    valid_count = sum(o is not None for o in outputs)
    signatures = [json.dumps(o, sort_keys=True) for o in outputs if o is not None]
    unique_count = len(set(signatures))
    print(f"Valid JSON: {valid_count}/5")
    print(f"Unique outputs: {unique_count} (1 = fully consistent)")
    for o in outputs:
        print(o)
    print()


--- Temperature 0 ---
Valid JSON: 5/5
Unique outputs: 1 (1 = fully consistent)
{'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
{'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
{'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
{'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
{'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}

--- Temperature 1.0 

### Part 4.3 — Hallucination probing

In [ ]:
# Test 1: ask about a detail NOT in a letter
test1_prompt = "Based on this letter, what is the applicant's credit score?\n\n" + LETTERS["L001"]
response1 = ask_llm(test1_prompt, system_prompt=SUMMARY_SYSTEM_V2, temperature=0)
print("--- Test 1: missing detail ---")
print(response1.text.strip())
print()

# Test 2: feed irrelevant text to the extractor
irrelevant_text = "Today's weather in Accra is sunny with a high of 31°C and light winds from the southwest."
result2 = extract_fields(irrelevant_text)
print("--- Test 2: irrelevant text ---")
print(result2)

--- Test 1: missing detail ---
The provided letter does not contain information regarding the applicant's credit score.

Akosua Mensah, a provision seller at Makola Market for 12 years, is requesting a loan of GHS 8,000 to purchase a deep freezer for business expansion. She reports a monthly profit of GHS 900 and has a two-year history of consistent contributions to the institution's susu scheme, totaling GHS 2,500. The applicant proposes a monthly repayment of GHS 450 over 20 months and has identified her sister, a teacher, as a guarantor.

--- Test 2: irrelevant text ---
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}


**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:**

1. My extraction accuracy was **high**. The hardest field was the **repayment amount**, because it was sometimes missing or written in different formats.

2. The reliability experiment showed that **temperature = 0** gives more consistent and predictable results, which is important for production systems.

3. Yes, the system could hallucinate when pressured to guess. Using instructions like **“use null, do not guess”** and validating the output with rules can reduce this risk.


### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:**

1. Applicants who write poorly in English but have good businesses could be unfairly judged because the system might misunderstand their applications and rate them as higher risk.

2. Sending personal data to a third-party API in another country creates **privacy and data-protection risks**. Before deployment, I would check data protection laws, the API provider’s security and privacy policies, where data is stored, and whether the institution has permission to send the data.

3. Two safeguards I would use are:

    **Human review:** A loan officer must review important or high-risk cases before a decision.
   **Monitoring and logging:** Keep records of system outputs and regularly check for errors or unfair patterns.


---
# Section 5 Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:**

 1) Prompting as engineering

Prompting is similar to tuning model hyperparameters because we change settings and test the results to improve performance. The difference is that prompting changes the instructions given to the model, while hyperparameters change how the model learns.

### 2) Trust

I would **not trust the system to run completely unattended** because loan decisions are high-stakes. The extraction accuracy and hallucination results were the most important evaluation results for me.

### 3)  Cost and scale

If one application uses about **1,000 tokens**, then 1,000 applications would need about **1,000,000 tokens per month**. This means I would compare providers based on cost, reliability, privacy, and performance.

### 4) Looking back at the course

Using an API is better here because it is faster, easier, and does not require collecting a large dataset or training a model from scratch. Training our own model could be better when we have a large specialized dataset, need more control, or have strict privacy requirements.


---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.